## 📂 오프라인 기능(Offline Features)

기능들이 이제 등록되었으므로, 우리는 이들을 사용하여 **오프라인 저장소**에서 정의된 기능을 가져와 **학습 데이터셋**을 생성할 수 있습니다.

이 예제에서 우리는 모든 사용 가능한 기능을 포함한 **전체 데이터셋**을 검색합니다. 하지만 이 노트북에서 탐색할 것처럼, 우리는 쉽게:
- **특정 기능**들을 전체 집합 대신 선택할 수 있습니다.
- **다른 FeatureView**들의 기능을 결합할 수 있습니다.
- **특정 시간 범위**를 기반으로 데이터를 필터링할 수 있습니다.

이 유연성은 우리의 머신러닝 모델의 필요에 맞게 데이터셋을 커스터마이징할 수 있게 합니다.

## 📦 의존성(Dependencies) 가져오기
Feast로 작업하기 전에 필요한 라이브러리를 가져와야 합니다:

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import feast
import pandas as pd
from datetime import datetime
import psycopg2
import yaml
import numpy as np

## ⚙️ Feature Store 설정 로드하기  

이전에 보았듯이, **Feast**와 상호작용하기 전에 `feature_store.yaml` 파일에서 설정을 로드해야 합니다. 이 파일은 레지스트리, 온라인 저장소 및 오프라인 저장소로의 연결을 포함한 Feature Store 설정 방식을 정의합니다.

In [ ]:
with open('feature_repo/feature_store.yaml', 'r') as file:
    fs_config_yaml = yaml.safe_load(file)

fs_config = feast.repo_config.RepoConfig(**fs_config_yaml)
fs = feast.FeatureStore(config=fs_config)

이 코드로 우리는 다음을 목표로 하고 있습니다:
- `feature_store.yaml`에서 **기능 저장소 설정**을 읽습니다.
- **FeatureStore** 객체를 초기화하여 기능 검색 및 관리를 활성화합니다.
- PostgreSQL 레지스트리 및 저장소 위치로의 **연결**을 설정합니다.

이 설정이 완료되면, 우리는 Feature Store에서 기능 데이터를 쿼리하기 시작할 수 있습니다!

### 시간 정확성(Point-in-time correctness)
Feast에서 데이터 포인트를 얻으려면, 우리는 기능을 원하는 ID와 날짜를 제공해야 합니다.  
그러면 Feast는 선택한 기능에 대해 해당 날짜 이전의 가장 최근 기능 값을 찾습니다. 이는 우리의 모델이 실수로 미래 정보를 사용하지 않도록 하여 데이터 누출을 방지합니다.  

우리의 경우, 우리는 song_rankings 데이터셋을 사용하여 song_properties 데이터에서 기능을 가져와 함께 병합합니다.  
rankings 데이터셋에서 같은 날짜와 같은 ID의 여러 노래가 있기 때문에, 우리는 또한 Feast가 중복을 폐기하지 않도록 각 항목에 작은 날짜 델타를 추가합니다.

In [ ]:
training_dataset = pd.read_parquet("../99-data_prep/song_rankings.parquet")
# Feast will remove rows with identical id and date so we add a small delta to each
microsecond_deltas = np.arange(0, len(training_dataset))*2
training_dataset['snapshot_date'] = training_dataset['snapshot_date'] + pd.to_timedelta(microsecond_deltas, unit='us')
training_dataset

우리가 확인한 것처럼, 우리는:
- **Parquet 데이터셋**을 Pandas DataFrame으로 로드합니다.
- 각 행에 마이크로초 델타를 추가하여 **타임스탬프 값의 고유성**을 보장합니다.
- Feast가 **엔티티 키와 타임스탬프**를 사용하여 기능을 매칭할 수 있도록 데이터셋을 준비합니다.

## 🏷️ 학습용 기능 선택하기  

이제 우리가 어떤 기능들을 원하는지 명시할 수 있으며, 또한 이러한 기능들이 어느 Feature View에서 오는지도 지정합니다.

In [ ]:
features=[
        "song_properties:is_explicit",
        "song_properties:duration_ms",
        "song_properties:danceability",
        "song_properties:energy",
        "song_properties:key",
        "song_properties:loudness",
        "song_properties:mode",
        "song_properties:speechiness",
        "song_properties:acousticness",
        "song_properties:instrumentalness",
        "song_properties:liveness",
        "song_properties:valence",
        "song_properties:tempo"
]

이 기능 목록은 다음을 수행합니다:
- `에너지`, `댄서빌리티` 및 `템포` 같은 **노래 관련 기능**을 지정합니다.
- **기능 명명 규칙**(`song_properties:<feature_name>`)을 사용하여 올바른 데이터를 검색합니다.
- **모델의 필요**에 기반하여 기능을 선택하는 데 있어 유연성을 허용합니다.

## 🔄 역사적 기능(Historical Features) 검색하기  

마지막으로, 우리는 Feast에서 우리의 데이터셋에 대한 역사적 기능 값을 검색합니다.

In [ ]:
training_df = fs.get_historical_features(entity_df=training_dataset, features=features).to_df()
training_df

마지막으로, 우리가 Feast에서 우리 데이터셋의 역사적 기능 값을 검색할 때, 우리는 다음을 수행합니다:
- 각 노래와 타임스탐프에 대한 기능을 검색하기 위해 **오프라인 저장소**에 쿼리합니다.
- 저장된 기능과 **엔티티 키**(예: 노래 ID)를 매칭합니다.
- 모델 학습에 쉽게 사용할 수 있도록 출력을 **Pandas DataFrame**으로 변환합니다.

이 단계들이 완료되었으므로, 이제 우리는 학습을 위해 준비된 **완전히 기능이 풍부한 데이터셋**을 갖게 되었습니다! 🚀

이제 우리는 Feast에서 학습 데이터를 얻는 방법을 알았으니, 추론(inference) 중 이를 사용하는 방법을 다음으로 살펴봅시다.  
다음 노트북에서 우리는 온라인 기능을 가져올 것입니다: [3-test_load_online_features.ipynb](3-test_load_online_features.ipynb)